# UCI Online Retail dataset
A real online retail transaction data set of two years.

## Additional Information
This Online Retail data set contains all the transactions occurring for a UK-based and registered, non-store online retail between 01/12/2009 and 09/12/2011.The company mainly sells unique all-occasion gift-ware. Many customers of the company are wholesalers.
Has missing values? Yes

## Additional Variable Information
|Header|Description|
|---|---|
|**InvoiceNo**| Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation. |
|**StockCode**| Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product. |
|**Description**| Product (item) name. Nominal. |
|**Quantity**| The quantities of each product (item) per transaction. Numeric.	|
|**InvoiceDate**| Invice date and time. Numeric. The day and time when a transaction was generated. |
|**UnitPrice**| Unit price. Numeric. Product price per unit in sterling (£). |
|**CustomerID**| Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer. |
|**Country**| Country name. Nominal. The name of the country where a customer resides.|

In [1]:
# from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# fetch dataset 
# online_retail = fetch_ucirepo(id=352) 

# data (as pandas dataframes) 
filename = "Online Retail"
orig_df = pd.read_excel(f"{filename}.xlsx")

In [53]:
from datetime import datetime

df = orig_df.copy()
df["Revenue"] = df["UnitPrice"] * df["Quantity"]
len_origdf = len(df)

# what row is this first
# order of priority: duplicate -> bad debt -> cancellation -> non-product -> write-off -> remaining: sales
stockcode_sales = [r'^\d{5}[A-Za-z]*$', "D", "S", r'^DCGS.*', r'^gift.*', "PADS"]
stockcode_expenses = ["POST", "DOT", "C2", "AMAZONFEE", "BANK CHARGES", "CRUK"]
stockcode_others = ["M", "m"]
conditions = [
				df["InvoiceDate"] >= datetime(2011, 12, 1), # exclude Dec 2011 as month is not complete
				df.duplicated(), # duplicates
				df["InvoiceNo"].astype("str").str.startswith("A"), # bad debt adjustments
                df["StockCode"].astype("str").str.fullmatch("|".join(stockcode_expenses)), # expenses
				df["InvoiceNo"].astype("str").str.startswith("C", na=False), # cancellations
				df["StockCode"].astype("str").str.fullmatch("|".join(stockcode_sales)), # all sales related: all products + discount + samples + dcgs + gift + pads
                (df["StockCode"].astype("str").str.fullmatch("|".join(stockcode_others)) & (df["CustomerID"].isna() == False)), # manual adjustment with customerid - likely to be custom orders or added orders from real customers
                (df["StockCode"].astype("str").str.fullmatch("|".join(stockcode_others)) & (df["CustomerID"].isna() == True)) # manual adjustment without customer id - likely to be adjustment of account balance, not related to sales
                
]

choices = ["dec_2011_data", "duplicates", "bad_debt", "expenses", "cancellations", "sales", "manual_with_customerid", "manual_without_customerid"]

df["type"] = np.select(conditions, choices, default="UNKNOWN")

# logging and checks
print(df[df["type"] == "UNKNOWN"])
assert (df["type"] == "UNKNOWN").sum() == 0 # ensure we have no unknown types

log = pd.DataFrame(df["type"].value_counts(normalize=True))
log = log.rename(columns={"proportion" : "proportion_of_rows"})
log["revenue_gbp"] = df.groupby("type")["Revenue"].sum()

keep = {
    "dec_2011_data": False,
    "sales": True,
	"cancellations": True,
    "manual_with_customerid": True,
	"duplicates": False,
    "expenses": False,
    "manual_without_customerid": False,
    "bad_debt": False
}

reasons = {
    "dec_2011_data": "Data for this period is incomplete, hence we exclude these data points",
	"sales": "All transactions, including successful transactions, discounts, samples, product related (DCGS), gifts, PADS, but excluding cancellations",
	"cancellations": "Include in revenue calculations as cancelled order offsets the corresponding successful transaction",
	"duplicates": "Assumed as double entry or system error",
    "expenses": "Costs incurred from postage, fees, shipment and online site charges; not sales",
    "manual_with_customerid": "Manual adjustment to account with real customerid, likely to be custom orders or added orders which are valid sales",
    "manual_without_customerid": "Manual adjustment of account without customerid, likely to be mere account adjustment that is not related to sales",
    "bad_debt": "Accounting adjustments are not sales"
}

log["keep_or_remove"] = log.index.map(keep)
log["reasons"] = log.index.map(reasons)
print(log)

Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country, Revenue, type]
Index: []
                           proportion_of_rows  revenue_gbp  keep_or_remove                                            reasons
type                                                                                                                         
sales                                0.923148  9633468.933            True  All transactions, including successful transac...
dec_2011_data                        0.047102   433668.010           False  Data for this period is incomplete, hence we e...
cancellations                        0.016045  -456959.170            True  Include in revenue calculations as cancelled o...
duplicates                           0.009206    20774.030           False            Assumed as double entry or system error
expenses                             0.003912    50543.471           False  Costs incurred from postage

In [54]:
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option('display.width', 1000)

# define our datasets
sales = df[(df["type"].isin(["sales", "cancellations", "manual_with_customerid"]))].copy()

# solve the issue of customerid being NaN;
# customers can checkout without having an account, so the customerid will be left blank
# since the InvoiceNo is unique, we can use InvoiceNo to infer the customers
sales["CustomerID"] = sales["CustomerID"].fillna("GUEST_" + sales["InvoiceNo"].astype("str"))
is_order = sales["type"] != "cancellations"
sales["Order"] = sales["InvoiceNo"].where(is_order) # NaN on cancellation rows
sales["Customer"] = sales["CustomerID"].where(is_order) # so cancelled guests aren't customers
assert sales["CustomerID"].isna().sum() == 0

# monthly revenue trend
monthly = (
    sales
    .set_index("InvoiceDate")
    .resample("1ME").agg({
        "Customer": "nunique",
        "Order": "nunique",
        "Quantity": "sum",
        "Revenue": "sum"
	})
)
monthly.index = monthly.index.strftime("%b-%Y")
print(monthly)

# revenue by region
region = (
    sales
    .groupby("Country").agg({
        "Revenue": "sum"
	})
    .sort_values("Revenue", ascending=False)
)
region = region[region.index != "United Kingdom"]
region_top9 = region.iloc[:10]
remainder = region.iloc[10:]
remainder = remainder.reset_index()
remainder["Country"] = "Others"
remainder = remainder.groupby("Country").agg({"Revenue": "sum"})
region = pd.concat([region_top9, remainder])
region = region.sort_values("Revenue", ascending=False)
print(region)

# Top 10 products
top_10_products = (
    sales
    .groupby("Description").agg({
        "Revenue": "sum"
	})
    .sort_values("Revenue", ascending=False)
    .head(10)
)
print(top_10_products)

# KPI cards
kpi_cards = {
	"net_revenue": f"£{monthly["Revenue"].sum():,.2f}",
	"total_orders": f"{int(monthly["Order"].sum()):,}",
	"total_customers": f"{int(sales["Customer"].nunique()):,}",
	"ave_order_val": f"£{monthly["Revenue"].sum() / monthly["Order"].sum():,.2f}",
}

             Customer  Order  Quantity      Revenue
InvoiceDate                                        
Dec-2010         1182   1694    340758   755104.710
Jan-2011          968   1212    306977   573772.530
Feb-2011          934   1173    277168   494690.960
Mar-2011         1313   1657    349853   672276.480
Apr-2011         1209   1501    288059   481636.771
May-2011         1347   1843    378550   719077.450
Jun-2011         1277   1677    340616   679384.850
Jul-2011         1261   1640    388752   662979.691
Aug-2011         1102   1439    405174   696453.190
Sep-2011         1495   1978    547585  1007727.831
Oct-2011         1701   2256    569359  1060084.160
Nov-2011         2021   3008    735736  1426302.970
               Revenue
Country               
Netherlands  271439.52
EIRE         247020.02
Others       202355.62
Germany      193843.77
France       175719.28
Australia    136922.50
Switzerland   52483.05
Spain         48660.60
Japan         35457.55
Belgium       35382

In [55]:
# excel dashboard
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import LineChart, BarChart, DoughnutChart, Reference
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.table import Table, TableStyleInfo

# helper
def move_df_to_worksheet(dataframes: list | pd.DataFrame, 
                         ws: openpyxl.worksheet.worksheet.Worksheet
						 ) -> openpyxl.worksheet.worksheet.Worksheet:
	if type(dataframes) == list:
		for df in dataframes:
			df = df.copy()
			df = df.reset_index()
			for row in dataframe_to_rows(df, index=False, header=True):
				ws.append(row)
			ws.append([]) # add an empty row in between different tables
	else:
		df = dataframes.copy()
		df = df.reset_index()
		for row in dataframe_to_rows(df, index=False, header=True):
			ws.append(row)
	return ws

def dashboard_title(cell_ref: openpyxl.cell.cell.Cell, 
                     value: any, 
					 font: Font, 
					 alignment: Alignment = Alignment(horizontal="center", vertical="center")
					 ) -> openpyxl.cell.cell.Cell:
	cell_ref.value = value
	cell_ref.font = font
	cell_ref.alignment = alignment
	return cell_ref

def dashboard_object(cell_ref: openpyxl.cell.cell.Cell, 
                     value: any, 
					 font: Font, 
					 alignment: Alignment = Alignment(horizontal="center", vertical="center")
					 ) -> openpyxl.cell.cell.Cell:
	cell_ref.value = value
	cell_ref.font = font
	cell_ref.alignment = alignment
	return cell_ref

def format_table(ws: openpyxl.worksheet.worksheet.Worksheet,
                 table_name: str,
                 table_start: str,
				 table_end: str):
	table_range = f"{table_start}:{table_end}"
	table = Table(displayName=table_name, ref=table_range)
	style = TableStyleInfo(
		name="TableStyleMedium4",
		showFirstColumn=False, # bold first col
    	showLastColumn=False, # bold last col
		showRowStripes=True, # alternating fill row
		showColumnStripes=False # alternating fill col
	)
	table.tableStyleInfo = style
	ws.add_table(table)
	return ws

def autofit(ws, cap=50):
	for col in ws.columns:
		longest = max(
			len(str(c.value)) for c in col if c.value is not None
		)
		ws.column_dimensions[col[0].column_letter].width = min(longest + 2, cap)

# create / initialise workbook and worksheets
wb = openpyxl.Workbook()
ws1 = wb.active
ws1.title = "Dashboard"
ws2 = wb.create_sheet("Log")
ws3 = wb.create_sheet("Data")

# move over our dataframes first
move_df_to_worksheet(log, ws2)
move_df_to_worksheet([monthly, region, top_10_products], ws3)

# format table log
log_lastcol_num = log.shape[1] + 1 # log itself has index so must +1
log_lastcol = get_column_letter(log_lastcol_num)
log_lastrow = len(log) + 1 # log itself has index so must +1
format_table(ws=ws2, table_name="log", table_start="A1", table_end=f"{log_lastcol}{log_lastrow}")
for i in range(2, 51): # reason column, E2 to E50 set to wrap text
	cell = ws2["E" + str(i)]
	cell.alignment = Alignment(wrap_text=True)

# format and design dashboard background FIRST
ws1.views.sheetView[0].showGridLines = False
canvasfill = PatternFill(start_color="F8F9FA", end_color="F8F9FA", fill_type="solid")
for row in ws1.iter_rows(1, 50, 1, 20):
	for cell in row:
		cell.fill = canvasfill
for col_i in range(21, 50):
	col_letter = openpyxl.utils.get_column_letter(col_i)
	ws1.column_dimensions[col_letter].hidden = True
for row_i in range(51, 100):
	ws1.row_dimensions[row_i].hidden = True

# setup title & KPI cards
tfont = Font(name="Segoe UI", size=20, bold=True, color="212529") # title
lfont = Font(name="Segoe UI", size=9, bold=True, color="6C757D") # card label
nfont = Font(name="Segoe UI", size=18, bold=True, color="212529") # card number
cardfill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid") # card fill

# title
thick = Side(style="thick", color="DEE2E6")
dashboard_object(ws1["B2"], "UCI Retail Sales KPI from Dec 2010 to Nov 2011¹", tfont)
ws1["B2"].border = Border(top=thick, left=None, right=None, bottom=thick)
ws1["B2"].fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
ws1.merge_cells("B2:S3")

# card 1 - 4
thin = Side(style="thin", color="DEE2E6")
for label, number in [("B5", "B6"), ("F5", "F6"), ("B13", "B14"), ("F13", "F14")]: # we just need the first cell of our cards / labels
    ws1[label].border = Border(top=thin, left=thin, right=thin)      # top half of the outline
    ws1[number].border = Border(bottom=thin, left=thin, right=thin)  # bottom half
    ws1[label].fill = ws1[number].fill = cardfill

dashboard_object(ws1["B5"], "NET REVENUE", lfont)
dashboard_object(ws1["B6"], kpi_cards["net_revenue"], nfont)
ws1.merge_cells("B5:D5")
ws1.merge_cells("B6:D11")

dashboard_object(ws1["F5"], "TOTAL ORDERS", lfont)
dashboard_object(ws1["F6"], kpi_cards["total_orders"], nfont)
ws1.merge_cells("F5:H5")
ws1.merge_cells("F6:H11")

dashboard_object(ws1["B13"], "TOTAL CUSTOMERS", lfont)
dashboard_object(ws1["B14"], kpi_cards["total_customers"], nfont)
ws1.merge_cells("B13:D13")
ws1.merge_cells("B14:D19")

dashboard_object(ws1["F13"], "AVERAGE ORDER VALUE", lfont)
dashboard_object(ws1["F14"], kpi_cards["ave_order_val"], nfont)
ws1.merge_cells("F13:H13")
ws1.merge_cells("F14:H19")

# region / country pie chart
# row references for each chart
mthrow_start = 1
mthrow_end = len(monthly) + 1
cdnutrow_start = mthrow_end + 2 # +2 instead because we will leave an empty row in between
cdnutrow_end = cdnutrow_start + len(region)
top10_start = cdnutrow_end + 2
top10_end = top10_start + len(top_10_products)

c_dnut = DoughnutChart()
c_dnut.title = "Revenue Breakdown by Region²"
dnut_data = Reference(ws3, min_col=2, min_row=cdnutrow_start, max_row=cdnutrow_end)
dnut_cat = Reference(ws3, min_col=1, min_row=cdnutrow_start+1, max_row=cdnutrow_end)
c_dnut.add_data(dnut_data, titles_from_data=True)
c_dnut.set_categories(dnut_cat)
c_dnut.holeSize = 55
c_dnut.width, c_dnut.height = 11.2, 8
ws1.add_chart(c_dnut, "B21")

# monthly revenue line chart
mth_line = LineChart()
mth_line.title = "Monthly Revenue trend"
mth_line.x_axis.title = "Month"
mth_line.y_axis.title = "Revenue"
mth_line.style = 13
mth_line.height = 8
mth_line.width = 15.8
mth_line.legend = None
mth_data = Reference(ws3, min_col=5, min_row=mthrow_start, max_row=mthrow_end)
mth_cat = Reference(ws3, min_col=1, min_row=mthrow_start+1, max_row=mthrow_end)
mth_line.add_data(mth_data, titles_from_data=True)
mth_line.set_categories(mth_cat)
ws1.add_chart(mth_line, "J5")

# add top 10 products bar chart
top10p = BarChart()
top10p.title = "Top 10 Products by Revenue"
top10p.x_axis.title = "Products"
top10p.y_axis.title = "Revenue"
top10p.style = 13
top10p.height = 8
top10p.width = 15.8
top10p.legend = None
top10p_data = Reference(ws3, min_col=2, min_row=top10_start, max_row=top10_end)
top10p_cat = Reference(ws3, min_col=1, min_row=top10_start+1, max_row=top10_end)
top10p.add_data(top10p_data, titles_from_data=True)
top10p.set_categories(top10p_cat)
ws1.add_chart(top10p, "J21")

format_table(ws=ws3, table_name="Monthly", table_start=f"A{mthrow_start}", table_end=f"E{mthrow_end}")
format_table(ws=ws3, table_name="TopTenProducts", table_start=f"A{top10_start}", table_end=f"B{top10_end}")
format_table(ws=ws3, table_name="Region", table_start=f"A{cdnutrow_start}", table_end=f"B{cdnutrow_end}")

# autofit ws2 and ws3 only
autofit(ws2)
autofit(ws3)

# footnotes
ws1["B36"] = "¹Exclude Dec 2011 because sales for this month is incomplete"
ws1["B37"] = "²Exclude United Kingdom because it accounts for roughly 85% of total revenue"

wb.save("Dashboard.xlsx")